In [4]:
import os
import json
import glob

DATA_DIR = "../disease_profiles"

relations = []

for file_path in glob.glob(os.path.join(DATA_DIR, "*.json")):

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    metadata = data.get("metadata", {})

    disease_id = metadata.get("disease_id")

    if disease_id is None:
        continue

    # =====================================================
    # CLINICAL FEATURES -> HAS_SYMPTOM
    # =====================================================

    clinical = data.get("clinical_features", {})

    for symptom in clinical.get("symptoms", []):

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "HAS_SYMPTOM",
            "t": symptom.lower().replace(" ", "_"),
            "t_type": "Symptom",
            "confidence": 1.0
        })

    # =====================================================
    # ABCDE RULE -> HAS_SYMPTOM
    # =====================================================

    abcde = clinical.get("abcde_rule", {})

    for value in abcde.values():

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "HAS_SYMPTOM",
            "t": value.lower().replace(" ", "_"),
            "t_type": "Symptom",
            "confidence": 1.0
        })

    # =====================================================
    # VISUAL CHARACTERISTICS -> HAS_VISUAL_FEATURE
    # =====================================================

    visual = clinical.get(
        "visual_characteristics",
        {}
    )

    for color in visual.get("colors", []):

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "HAS_VISUAL_FEATURE",
            "t": color.lower().replace(" ", "_"),
            "t_type": "VisualFeature",
            "confidence": 1.0
        })

    for shape in visual.get("shapes", []):

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "HAS_VISUAL_FEATURE",
            "t": shape.lower().replace(" ", "_"),
            "t_type": "VisualFeature",
            "confidence": 1.0
        })

    for border in visual.get("borders", []):

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "HAS_VISUAL_FEATURE",
            "t": border.lower().replace(" ", "_"),
            "t_type": "VisualFeature",
            "confidence": 1.0
        })

    # =====================================================
    # ETIOLOGY -> HAS_RISK
    # =====================================================

    etiology = data.get("etiology", {})

    for risk in etiology.get("risk_factors", []):

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "HAS_RISK",
            "t": risk.lower().replace(" ", "_"),
            "t_type": "RiskFactor",
            "confidence": 1.0
        })

    # =====================================================
    # ETIOLOGY -> CAUSED_BY
    # =====================================================

    for cause in etiology.get("causes", []):

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "CAUSED_BY",
            "t": cause.lower().replace(" ", "_"),
            "t_type": "Cause",
            "confidence": 1.0
        })

    # =====================================================
    # DIAGNOSIS -> DIAGNOSED_BY
    # =====================================================

    diagnosis = data.get("diagnosis", {})

    for values in diagnosis.values():

        if isinstance(values, list):

            for item in values:

                relations.append({
                    "s": disease_id,
                    "s_type": "Disease",
                    "r": "DIAGNOSED_BY",
                    "t": item.lower().replace(" ", "_"),
                    "t_type": "DiagnosticFeature",
                    "confidence": 1.0
                })

    # =====================================================
    # TREATMENT -> TREATED_BY
    # =====================================================

    treatment = data.get("treatment", {})

    for values in treatment.values():

        if isinstance(values, list):

            for item in values:

                relations.append({
                    "s": disease_id,
                    "s_type": "Disease",
                    "r": "TREATED_BY",
                    "t": item.lower().replace(" ", "_"),
                    "t_type": "Treatment",
                    "confidence": 1.0
                })

    # =====================================================
    # PROGNOSIS -> HAS_PROGNOSIS
    # =====================================================

    prognosis = data.get("prognosis", {})

    outlook = prognosis.get(
        "overall_outlook"
    )

    if outlook:

        relations.append({
            "s": disease_id,
            "s_type": "Disease",
            "r": "HAS_PROGNOSIS",
            "t": outlook.lower().replace(" ", "_"),
            "t_type": "Prognosis",
            "confidence": 1.0
        })

# =====================================================
# REMOVE DUPLICATES
# =====================================================

unique_relations = []

seen = set()

for rel in relations:

    key = (
        rel["s"],
        rel["r"],
        rel["t"]
    )

    if key not in seen:

        seen.add(key)

        unique_relations.append(rel)

# =====================================================
# SAVE
# =====================================================

with open(
    "relations_auto.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        unique_relations,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 60)
print("TOTAL RELATIONS:", len(unique_relations))
print("=" * 60)

print("\nSAMPLE:\n")

for item in unique_relations[:10]:
    print(item)

TOTAL RELATIONS: 159

SAMPLE:

{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_SYMPTOM', 't': 'itching', 't_type': 'Symptom', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_SYMPTOM', 't': 'burning', 't_type': 'Symptom', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_SYMPTOM', 't': 'bleeding', 't_type': 'Symptom', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_SYMPTOM', 't': 'crusting', 't_type': 'Symptom', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_VISUAL_FEATURE', 't': 'pink', 't_type': 'VisualFeature', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_VISUAL_FEATURE', 't': 'red', 't_type': 'VisualFeature', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_VISUAL_FEATURE', 't': 'brown', 't_type': 'VisualFeature', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r': 'HAS_RISK', 't': 'fair_skin', 't_type': 'RiskFactor', 'confidence': 1.0}
{'s': 'AKIEC', 's_type': 'Disease', 'r'

In [6]:
import json
from collections import defaultdict

with open("relations_auto.json","r") as f:
    relations = json.load(f)

WEIGHTS = {
    "HAS_SYMPTOM":5,
    "HAS_VISUAL_FEATURE":4,
    "HAS_RISK":2,
    "CAUSED_BY":1,
    "DIAGNOSED_BY":1,
    "TREATED_BY":0,
    "HAS_PROGNOSIS":0
}

def reason(features):

    scores = defaultdict(float)

    for edge in relations:

        if edge["t"] in features:

            scores[edge["s"]] += WEIGHTS.get(
                edge["r"],
                0
            )

    return sorted(
        scores.items(),
        key=lambda x:x[1],
        reverse=True
    )
reason([
    "color_variation",
    "asymmetry",
    "irregular"
])

[('MEL', 15.0)]

In [6]:
import json
from collections import defaultdict

from semantic_feature_retriever import (
    SemanticFeatureRetriever
)

# ==========================================================
# LOAD RELATIONS
# ==========================================================

with open(
    "relations_auto.json",
    "r",
    encoding="utf-8"
) as f:

    relations = json.load(f)

# ==========================================================
# LOAD SEMANTIC RETRIEVER
# ==========================================================

semantic_retriever = SemanticFeatureRetriever(
    catalog_path="semantic_features/feature_catalog.json",
    index_path="semantic_features/feature_index.faiss"
)

semantic_retriever.load_index()

# ==========================================================
# RELATION WEIGHTS
# ==========================================================

WEIGHTS = {

    "HAS_SYMPTOM": 5,

    "HAS_VISUAL_FEATURE": 4,

    "HAS_RISK": 2,

    "CAUSED_BY": 1,

    "DIAGNOSED_BY": 1,

    "TREATED_BY": 0,

    "HAS_PROGNOSIS": 0
}

# ==========================================================
# FEATURE ALIASES (RULE-BASED)
# ==========================================================

FEATURE_MAP = {

    # MEL

    "asymmetry": "asymmetry",
    "asymmetrical": "asymmetry",

    "irregular border": "border_irregularity",
    "uneven border": "border_irregularity",
    "jagged border": "border_irregularity",

    "color variation": "color_variation",
    "multiple colors": "color_variation",
    "different colors": "color_variation",

    "growing lesion": "growth",
    "growth": "growth",

    "bleeding": "bleeding",

    "itching": "itching",

    # BCC

    "pearly nodule": "pearly_nodule",
    "shiny bump": "pearly_nodule",

    "ulcer": "ulceration",
    "ulceration": "ulceration",

    # AKIEC

    "scaly patch": "scaly_patch",

    "rough patch": "rough_texture",
    "rough texture": "rough_texture",

    # NV

    "uniform color": "uniform_color",

    "symmetrical": "symmetrical_shape",
    "regular shape": "symmetrical_shape",

    # BKL

    "waxy": "waxy_surface",
    "waxy surface": "waxy_surface",

    "stuck on": "stuck_on_appearance",
    "stuck-on": "stuck_on_appearance",

    # DF

    "dimple sign": "dimple_sign",

    # VASC

    "red papule": "red_papule",
    "red bump": "red_papule"
}

# ==========================================================
# FEATURE EXTRACTION
# ==========================================================

def extract_features(query):

    query_lower = query.lower()

    # --------------------------
    # Rule Features
    # --------------------------

    rule_features = set()

    for phrase, feature in FEATURE_MAP.items():

        if phrase in query_lower:

            rule_features.add(
                feature
            )

    # --------------------------
    # Semantic Features
    # --------------------------

    semantic_results = semantic_retriever.retrieve(
        query=query,
        top_k=10,
        threshold=0.45
    )

    semantic_features = set()

    for item in semantic_results:

        semantic_features.add(
            item["feature"]
        )

    # --------------------------
    # Merge
    # --------------------------

    final_features = list(

        rule_features.union(
            semantic_features
        )

    )

    return {

        "final_features":
            final_features,

        "rule_features":
            list(rule_features),

        "semantic_features":
            list(semantic_features),

        "semantic_results":
            semantic_results
    }

# ==========================================================
# GRAPH REASONING
# ==========================================================

def graph_reasoning(features):

    scores = defaultdict(float)

    explanations = defaultdict(list)

    for edge in relations:

        target = edge["t"]

        if target not in features:
            continue

        disease = edge["s"]

        score = (

            WEIGHTS.get(
                edge["r"],
                0
            )

            *

            edge["confidence"]

        )

        scores[disease] += score

        explanations[disease].append({

            "feature":
                target,

            "relation":
                edge["r"],

            "score":
                round(
                    score,
                    2
                )
        })

    return scores, explanations

# ==========================================================
# QUERY -> DISEASE
# ==========================================================

from explanation_engine import (
    ExplanationEngine
)

explainer = ExplanationEngine(
    knowledge_dir="../disease_profiles"
)

def predict_from_query(query):

    feature_info = extract_features(
        query
    )

    features = feature_info[
        "final_features"
    ]

    scores, explanations = graph_reasoning(
        features
    )

    ranked = sorted(

        scores.items(),

        key=lambda x: x[1],

        reverse=True
    )

    # ======================================================
    # DISPLAY
    # ======================================================

    print("=" * 60)
    print("QUERY")
    print("=" * 60)
    print(query)

    print("\nRULE FEATURES")
    print("-" * 60)
    print(
        feature_info[
            "rule_features"
        ]
    )

    print("\nSEMANTIC FEATURES")
    print("-" * 60)
    print(
        feature_info[
            "semantic_features"
        ]
    )

    print("\nFINAL FEATURES")
    print("-" * 60)
    print(features)

    print("\nSEMANTIC MATCHES")
    print("-" * 60)

    for item in feature_info[
        "semantic_results"
    ]:

        print(
            f"{item['feature']:25s}"
            f"{item['score']:.3f}"
        )

    print("\nRANKING")
    print("=" * 60)

    for disease, score in ranked:

        print(
            f"{disease:10s}"
            f" -> "
            f"{score:.2f}"
        )

    print("\nEXPLANATION")
    print("=" * 60)

    for disease, _ in ranked:

        print(f"\n{disease}")

        for item in explanations[disease]:

            print(
                f"  {item['feature']} "
                f"({item['relation']}) "
                f"+{item['score']}"
            )

    # ======================================================
    # DISEASE KNOWLEDGE EXPLANATION
    # ======================================================

    if ranked:

        top_disease = ranked[0][0]
        top_score = ranked[0][1]

        explanation = explainer.generate(
            disease_id=top_disease,
            disease_score=top_score,
            reasoning_details=
                explanations[top_disease]
        )

        print("\n")
        explainer.display(
            explanation
        )

    return ranked

# ==========================================================
# TEST
# ==========================================================

predict_from_query(
    "The lesion has uneven edges and many colors"
)

Loading SentenceTransformer...
QUERY
The lesion has uneven edges and many colors

RULE FEATURES
------------------------------------------------------------
[]

SEMANTIC FEATURES
------------------------------------------------------------
['border_irregularity', 'scaly_patch', 'red_papule', 'symmetrical_shape', 'asymmetry', 'pearly_nodule']

FINAL FEATURES
------------------------------------------------------------
['border_irregularity', 'scaly_patch', 'red_papule', 'symmetrical_shape', 'asymmetry', 'pearly_nodule']

SEMANTIC MATCHES
------------------------------------------------------------
asymmetry                0.554
red_papule               0.539
scaly_patch              0.536
border_irregularity      0.505
pearly_nodule            0.491
symmetrical_shape        0.461

RANKING
MEL        -> 11.00

EXPLANATION

MEL
  asymmetry (HAS_SYMPTOM) +5.0
  border_irregularity (HAS_SYMPTOM) +5.0
  asymmetry (DIAGNOSED_BY) +1.0


KeyError: 'common_locations'